# Hakam — training v2

Fixes found after the first run: augmentation was identical every epoch, and the model memorised the training set.

1. **Runtime → Change runtime type → L4 GPU**
2. **Runtime → Run all** (about 30 minutes)

## 1. GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU - change the runtime type."
print(torch.cuda.get_device_name(0))

## 2. Code from GitHub, data and labels from Drive

In [ ]:
import shutil, zipfile
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
pkg = Path("/content/drive/MyDrive/hakam_colab")
work = Path("/content/hakam")
if not work.exists():
    !git clone -q https://github.com/FerasMad/hakam.git /content/hakam

for split in ["Train", "Valid", "Test"]:
    local = Path("/content") / f"{split}.zip"
    shutil.copy(pkg / "data" / f"{split}.zip", local)
    zipfile.ZipFile(local).extractall(work / "data" / "mvfouls")
    local.unlink()
    print(split, "ready")

labels = work / "artifacts" / "preprocessing" / "private"
labels.mkdir(parents=True, exist_ok=True)
for f in (pkg / "manifests").glob("*.csv"):
    shutil.copy(f, labels)
print("labels ready")

## 3. Install

In [ ]:
%cd /content/hakam
!pip install -q transformers ultralytics

## 4. Cache frames (last clip of each incident)

In [ ]:
!python scripts/cache_frames.py --splits train valid test

## 5. Find the contact point in each clip
A pretrained person detector finds the closest pair of players. Their midpoint centres the crop in Experiment 4.

In [ ]:
!python scripts/zoom_boxes.py --splits train valid test

## 6. Experiment 3 — fixed augmentation + regularisation
Fresh augmentation every epoch, 9 of 12 blocks frozen, smaller learning rate, label smoothing, weight decay, dropout.

In [ ]:
!python -m src.models.train --mode finetune --stage card --geometry resize --augment mild_aug_v1 --freeze-blocks 9 --lr 1e-5 --epochs 6 --weight-decay 0.1 --label-smoothing 0.1 --head-dropout 0.3 --batch-size 8 --num-workers 2 --save --name card_exp3_regularised

## 7. Experiment 4 — crop centred on the contact
Same as Experiment 3, but a square crop follows the foul: no stretching, and the foul is never cut out.

In [ ]:
!python -m src.models.train --mode finetune --stage card --geometry zoom --augment mild_aug_v1 --freeze-blocks 9 --lr 1e-5 --epochs 6 --weight-decay 0.1 --label-smoothing 0.1 --head-dropout 0.3 --batch-size 8 --num-workers 2 --save --name card_exp4_contact_crop

## 8. Results and save to Drive
The CI resamples whole matches. Two runs differ only if their intervals don't overlap.

In [ ]:
!python scripts/summarise_runs.py --save /content/drive/MyDrive/hakam_colab/runs